In [ ]:
import os
import torch
from pathlib import Path

# 1. 🛑 ป้องกัน Colab ค้าง
os.environ["TQDM_DISABLE"] = "1"

from anomalib.data import MVTecAD
from anomalib.models import EfficientAd
from anomalib.models.image.efficient_ad.torch_model import EfficientAdModelSize
from anomalib.engine import Engine
from lightning.pytorch.callbacks import (
    TQDMProgressBar,
    Callback,
    EarlyStopping,
    ModelCheckpoint
)
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch import seed_everything # 🌟 เพิ่ม Import ตรงนี้แล้วครับ!

# 🌟 ล็อก Random Seed เพื่อให้รันกี่ครั้งผลลัพธ์ก็เหมือนเดิม (สำคัญมากสำหรับงานวิจัย)
seed_everything(42, workers=True)

# ============================================================
# 🌟 Callback: ตัวรายงานผลแบบ Real-time (เพิ่มการดูค่า Validation)
# ============================================================
class MasterMetricCallback(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        t_loss = metrics.get("train_loss")
        v_loss = metrics.get("val_loss")

        # ป้องกันการปริ้นท์ตอน Sanity Check (Epoch 0)
        if trainer.current_epoch > 0:
            epoch_info = f"🔄 Epoch {trainer.current_epoch:>3}/{trainer.max_epochs}"
            t_loss_info = f"📉 Train Loss: {t_loss.item():.4f}" if t_loss is not None else "Train Loss: N/A"
            v_loss_info = f"📊 Val Loss: {v_loss.item():.4f}" if v_loss is not None else "📊 Val Loss: N/A"

            print(f"{epoch_info} | {t_loss_info} | {v_loss_info}")

# ============================================================
# 🚀 เริ่มต้นกระบวนการ
# ============================================================
print("🚀 เริ่มต้นกระบวนการเทรน EfficientAD (Ultra-Optimized)...")

# 1. 📦 เตรียมข้อมูล (Dataset)
datamodule = MVTecAD(
    root="/data_scan/dataset",
    category="mvtec_dataset",
    train_batch_size=1,   # กฎเหล็กของ EfficientAD
    eval_batch_size=8,
    num_workers=4,
    # 🌟 Optimize: ดึง Test Set มาทำ Validation เพื่อวัดผลระหว่างเทรน
    val_split_mode="from_test",
    val_split_ratio=0.5
)

# 2. 🤖 เตรียมโมเดล (Model)
model = EfficientAd(
    model_size=EfficientAdModelSize.S,
    lr=2e-4,
    weight_decay=1e-5
)

# 3. 📁 ระบบบันทึกผล (Logger & Checkpoints)
logger = CSVLogger(save_dir="/content/results", name="efficient_ad_logs")

# 🌟 Optimize: กลับมาใช้ train_loss เพื่อความเสถียร (ไม่แครช)
early_stopping = EarlyStopping(
    monitor="train_loss",  
    patience=15,
    mode="min",
    min_delta=0.0001
)

checkpoint = ModelCheckpoint(
    dirpath="/content/results/checkpoints",
    filename="efficientad-best-{epoch:02d}-{train_loss:.4f}",
    monitor="train_loss",  
    mode="min",
    save_top_k=1
)

# 4. ⚙️ เครื่องยนต์ควบคุมการเทรน (Engine)
engine = Engine(
    max_epochs=100,
    accelerator="gpu",
    devices=1,
    precision="16-mixed", # 🌟 Optimize: ใช้ Mixed Precision เร็วขึ้น 2 เท่า กิน VRAM ลดลงครึ่งนึง!
    logger=logger,
    callbacks=[
        TQDMProgressBar(refresh_rate=0),
        MasterMetricCallback(),
        early_stopping,
        checkpoint
    ]
)

# 5. 🔥 เริ่มการเทรน (Fit)
print("🧠 AI กำลังจดจำความสมบูรณ์แบบของชิ้นงาน...")
os.environ.pop("TQDM_DISABLE", None)
engine.fit(datamodule=datamodule, model=model)

# 6. 🎯 ทำข้อสอบไฟนอล (Test)
print("\n" + "="*50)
print(f"🏆 เทรนเสร็จสิ้น! บันทึกโมเดลที่ดีที่สุดไว้ที่: {checkpoint.best_model_path}")
print("="*50 + "\n")

print("🎯 กำลังทดสอบประสิทธิภาพด้วยภาพ 'ของเสีย' (รอยขีดข่วน/รอยบุบ)...")
engine.test(datamodule=datamodule, model=model)

print("✅ ทุกขั้นตอนเสร็จสมบูรณ์!")

Seed set to 42


🚀 เริ่มต้นกระบวนการเทรน EfficientAD (Ultra-Optimized)...
🧠 AI กำลังจดจำความสมบูรณ์แบบของชิ้นงาน...


OSError: [WinError 1314] A required privilege is not held by the client: 'D:\\IAD\\results\\EfficientAd\\MVTecAD\\mvtec_dataset\\v0' -> 'D:\\IAD\\results\\EfficientAd\\MVTecAD\\mvtec_dataset\\latest'